In [0]:
%run ./01_setup_environment

In [0]:

# ========================================
# 06_silver_claims_transformation
# ========================================

from pyspark.sql.functions import *

try:

    claims_df = spark.read.format("delta") \
        .load(f"{bronze_path}/claims")

    silver_claims_df = claims_df.dropDuplicates(["claim_id"])

    silver_claims_df = silver_claims_df.withColumn(
        "claim_amount",
        round(col("claim_amount"), 2)
    )

    silver_claims_df = silver_claims_df.withColumn(
        "claim_status",
        upper(trim(col("claim_status")))
    )

    silver_claims_df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{silver_path}/claims_clean")

    log_audit(
        "claims_pipeline",
        "silver",
        "claims_clean",
        silver_claims_df.count(),
        "SUCCESS"
    )

    print("Silver Claims Transformation Completed")

except Exception as e:

    log_audit(
        "claims_pipeline",
        "silver",
        "claims_clean",
        0,
        "FAILED",
        str(e)
    )

    raise e